Hyperparams selection of SVC models

# Libraries

In [1]:
import openpyxl
import pandas as pd
from pathlib import Path
import numpy as np
import sys
from tqdm import tqdm
import optuna
import joblib
import json
from pathlib import Path
import os
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [3]:
# hyperparams
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import *

In [4]:
# sklearn models pipeline
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_utils import MLPipeline

In [5]:
RANDOM_STATE = 42

# Datatasets selection

In [6]:
path_data = Path('../data')
path_results = Path('../results/')
path_highlighted = path_results / 'metrics_modelling2_highlight.xlsx'

In [7]:
df_filtered = pd.DataFrame()
for i, page in enumerate(['Net Impact', 'Splashing']):
    df_highlighted = pd.read_excel(path_highlighted, sheet_name=page)
    wb = openpyxl.load_workbook(path_highlighted, data_only=True)
    ws = wb.worksheets[i]

    highlighted_rows = []
    for row in range(1, ws.max_row):
        cell = ws.cell(column=1, row=row)
        bgColor = cell.fill.bgColor.index
        fgColor = cell.fill.fgColor.index
        if bgColor != '00000000':
            highlighted_rows.append(row)
    highlighted_rows = np.array(highlighted_rows) - 2
    df_temp = df_highlighted.iloc[highlighted_rows]
    df_filtered = pd.concat((df_filtered, df_temp))
df_filtered.reset_index(drop=True, inplace=True)
df_filtered = df_filtered.loc[
    df_filtered['model'].apply(lambda x: 'svc' in x) &
    df_filtered['dataset'].apply(lambda x: 'multicollinearity' not in x)
    ]

Datasets for parameters selection.

In [8]:
sorted(df_filtered['model'].unique())

['svc_smote_net_impact_ordenc',
 'svc_smote_splashing_onehot',
 'svc_smote_splashing_ordenc']

# Parameters selection

In [9]:
path_best_model = Path('../results/svc_retrain_modelling_3')
if not os.path.exists(path_best_model): os.makedirs(path_best_model)

In [10]:
def get_params(trial, model_str, random_state):
    if 'svc' in model_str:
        params = {
            'random_state': random_state,
            'C': trial.suggest_float('C', 1e-5, 1e5, log=True),
            'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'sigmoid']),
            'gamma': trial.suggest_float('gamma', 1e-5, 10, log=True),
            'coef0': trial.suggest_float('coef0', -1.0, 1.0)}
    return params

In [11]:
def objective(
        trial, train, test, target, model_str, 
        random_state, dataset_filename, kernel, cat_features=['wettability']):
    params = get_params(trial, model_str, random_state)
    params['kernel'] = kernel
    model = get_model(model_str, params)
    
    preproc_pipeline = get_preproc_pipeline(model_str=model_str, random_state=random_state, 
                                            cat_features=cat_features, 
                                            num_features=dict_num_features[dataset_filename])
    pipeline = [('model', model)]
    if preproc_pipeline: pipeline.insert(0, preproc_pipeline[0])
    pipeline = Pipeline(steps=pipeline)
    pipeline.fit(train.drop(columns=[target]), train[target])

    pipeline = [('model', model)]
    if preproc_pipeline: pipeline.insert(0, preproc_pipeline[0])
    preproc_pipeline = [x for x in preproc_pipeline if x[0]!='smt']
    pipeline = Pipeline(steps=pipeline)
    preds = pipeline.predict(test.drop(columns=[target]))
    f1 = f1_score(test[target], preds)
    return f1

In [12]:
df_optuna = pd.DataFrame()
dict_results = {}
for i in (pbar:=tqdm(range(df_filtered.shape[0]))):
    dataset_filename = df_filtered.iloc[i]['dataset']
    if 'df_full' in dataset_filename: continue
    f1_global = 0
    model_str = df_filtered.iloc[i]['model']
    
    target = df_filtered.iloc[i]['target']
    train, test = get_train_test(
        dataset_filename=dataset_filename,
        target=target)
    for kernel in ['linear', 'poly']:
        pbar.set_description(f"Processing\t{model_str}\t{dataset_filename}\t{kernel}")
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize")
        def obj(trial):
                return objective(
                trial, train, test, 
                target, model_str, dataset_filename=dataset_filename, 
                random_state=RANDOM_STATE, cat_features=['wettability'], kernel=kernel)
        study.optimize(obj, n_trials=1500, show_progress_bar=True, n_jobs=os.cpu_count()-2)

        params = study.best_params
        model = get_model(model_str, params)
        preproc_pipeline = get_preproc_pipeline(model_str=model_str, random_state=RANDOM_STATE, 
                                                cat_features=['wettability'], 
                                                num_features=dict_num_features[dataset_filename])
        preproc_pipeline = [x for x in preproc_pipeline if x[0]!='smt']
        pipeline = [('model', model)]
        if preproc_pipeline: pipeline.insert(0, preproc_pipeline[0])
        pipeline = Pipeline(steps=pipeline)
        pipeline.fit(train.drop(columns=[target]), train[target])
        preds = pipeline.predict(test.drop(columns=[target]))
        joblib.dump(pipeline, str(path_best_model / f'{model_str}_{dataset_filename}_{target}_{kernel}.pkl'))
        metrics = [{
                'accuracy': accuracy_score(test[target], preds),
                'f1': f1_score(test[target], preds),
                'precision': precision_score(test[target], preds),
                'recall': recall_score(test[target], preds),
                'roc_auc': roc_auc_score(test[target], preds)}]
        dict_results[f'{model_str}_{dataset_filename}'] = {
                'dataset': dataset_filename,
                'model': model_str,
                'target': target,
                'metrics': metrics,
                'params': study.best_params}
        df_current = pd.DataFrame({
                'dataset': [dataset_filename],
                'target': [target],
                'model': [f'{model_str}'.replace(f'_{dataset_filename}', '')],
                'kernel': kernel,
                'optuna_flg': [1],
                'accuracy': [metrics[0]['accuracy']],
                'f1': [metrics[0]['f1']],
                'precision': [metrics[0]['precision']],
                'recall': [metrics[0]['recall']],
                'roc_auc': [metrics[0]['roc_auc']]})
        df_optuna = pd.concat((df_optuna, df_current))
        clear_output()
        metrics = []
df_results = pd.concat((df_filtered, df_optuna))

Processing	svc_smote_splashing_ordenc	df_modelling_dimensionless	poly: 100%|██████████| 5/5 [18:04<00:00, 216.89s/it]


In [13]:
with open("../results/optuna_results_svc_2.json", "w") as outfile: 
    json.dump(dict_results, outfile)
df_results.reset_index(drop=True, inplace=True)
df_results.to_excel('../results/metrics_modelling2_svc_2.xlsx')

In [14]:
df_results[df_results['optuna_flg']==1].sort_values(by='f1', ascending=False)

,dataset,target,model,accuracy,f1,precision,recall,roc_auc,kernel,optuna_flg
12,df_modelling_dimensionless_pf,splashing,svc_smote_splashing_onehot,0.866667,0.900000,0.865385,0.937500,0.839120,poly,1.0
14,df_modelling_dimensionless,splashing,svc_smote_splashing_ordenc,0.813333,0.854167,0.854167,0.854167,0.797454,poly,1.0
9,df_modelling_dimensionless,splashing,svc_smote_splashing_onehot,0.786667,0.846154,0.785714,0.916667,0.736111,linear,1.0
10,df_modelling_dimensionless,splashing,svc_smote_splashing_onehot,0.786667,0.829787,0.847826,0.812500,0.776620,poly,1.0
7,df_modelling_dimensionless_pf,net_impact,svc_smote_net_impact_ordenc,0.906667,0.820513,0.842105,0.800000,0.872727,linear,1.0
8,df_modelling_dimensionless_pf,net_impact,svc_smote_net_impact_ordenc,0.893333,0.800000,0.800000,0.800000,0.863636,poly,1.0
6,df_modelling_dimensionless,net_impact,svc_smote_net_impact_ordenc,0.880000,0.780488,0.761905,0.800000,0.854545,poly,1.0
11,df_modelling_dimensionless_pf,splashing,svc_smote_splashing_onehot,0.640000,0.780488,0.640000,1.000000,0.500000,linear,1.0
13,df_modelling_dimensionless,splashing,svc_smote_splashing_ordenc,0.573333,0.686275,0.648148,0.729167,0.512731,linear,1.0
5,df_modelling_dimensionless,net_impact,svc_smote_net_impact_ordenc,0.733333,0.000000,0.000000,0.000000,0.500000,linear,1.0
